## 1. Objective

This notebook builds a REST API using FastAPI to serve churn predictions. The API will:

- Accept customer features as JSON
- Return churn probability
- Be testable via Swagger UI


2. Load Model & Dependencies

In [1]:
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
import uvicorn

# Load model
model = joblib.load("../models/logreg_model.pkl")  # or xgb_model.pkl


3. Define Input Schema

In [2]:
class CustomerFeatures(BaseModel):
    tenure: float
    MonthlyCharges: float
    TotalCharges: float
    Contract_Two_year: int
    InternetService_Fiber_optic: int
    OnlineSecurity_No: int
    TechSupport_No: int
    PaperlessBilling: int
    # Add other relevant encoded features as needed


4. Create FastAPI App

In [3]:
app = FastAPI(
    title="Telco Churn Predictor",
    description="API that predicts customer churn probability",
    version="1.0"
)


5. Define /predict Endpoint

In [4]:
@app.post("/predict")
def predict_churn(data: CustomerFeatures):
    input_data = np.array([[
        data.tenure,
        data.MonthlyCharges,
        data.TotalCharges,
        data.Contract_Two_year,
        data.InternetService_Fiber_optic,
        data.OnlineSecurity_No,
        data.TechSupport_No,
        data.PaperlessBilling
        # Add other features in correct order
    ]])

    probability = model.predict_proba(input_data)[0][1]
    prediction = model.predict(input_data)[0]

    return {
        "churn_probability": round(float(probability), 4),
        "prediction": int(prediction),
        "interpretation": "Likely to churn" if prediction == 1 else "Likely to stay"
    }


6. Run API Locally

## Summary

- Built a FastAPI app with a `/predict` endpoint
- Accepts customer features and returns churn probability
- Ready for containerization and cloud deployment


